In [1]:
#this notebook is interested in doing simple rnn from scratch on pytorch since i dont really understand sequential models well

#data is Lorenz Attractor Trajectory Forecasting


#it predicts behavior of physical systems susceptible to chaos like fluid convection/weather systems using a set of differential equations



#euler method numerical integration method for dataset collection

x_initial=1.0
y_initial=1.0
z_initial=1.0
dt=0.01

list_of_x=[x_initial]
list_of_y=[y_initial]
list_of_z=[z_initial]
for i in range(9999):

  dx=10*(y_initial-x_initial)*dt
  dy=(x_initial*(28-z_initial)-y_initial)*dt
  dz=(x_initial*y_initial-(8/3)*z_initial)*dt

  x_initial+=dx
  y_initial+=dy
  z_initial+=dz

  list_of_x.append(x_initial)
  list_of_y.append(y_initial)
  list_of_z.append(z_initial)
print(list_of_x)
print(list_of_y)
print(list_of_z)

total_x=sum(list_of_x)
total_y=sum(list_of_y)
total_z=sum(list_of_z)

mean_x=total_x/len(list_of_x)
mean_y=total_y/len(list_of_y)
mean_z=total_z/len(list_of_z)



[1.0, 1.0, 1.026, 1.0751566666666668, 1.1456131764000002, 1.236317170692351, 1.346883543428101, 1.4774923478684758, 1.6288142132177872, 1.8019572717941623, 1.998430804872221, 2.220121683732517, 2.4692802570439603, 2.748512641279831, 3.0607764129637878, 3.4093764662250017, 3.7979572522780254, 4.230486703839468, 4.711225790837451, 5.2446757591584845, 5.835492566428736, 6.488354752059635, 7.207766916232549, 7.997776210491928, 8.861574088902051, 9.800950821048652, 10.815567490076651, 11.902012164606651, 13.052618140640684, 14.254049243671066, 15.485708870384482, 16.718115085128414, 17.911509580079926, 19.01512781856343, 19.96772045447606, 20.700011599171194, 21.139691785369237, 21.21913659552547, 20.8852397709095, 20.109666994774575, 18.89688574723792, 17.287144888755957, 15.352659499629421, 13.187442363675881, 10.893524789236814, 8.567457514809934, 6.290380611211302, 4.123128629880224, 2.105941502003087, 0.26124794606816404, -1.402176878776669, -2.8849607994028545, -4.193505440814818, -5.

In [2]:
import torch
tensor_2d=torch.stack([torch.tensor(list_of_x), torch.tensor(list_of_y), torch.tensor(list_of_z)], dim=1)
amount_of_entries=len(list_of_y)/50
tensor_reshaped=tensor_2d.reshape(int(amount_of_entries), 50, 3)



input_data=tensor_reshaped[:, :-1, :]
output_data=tensor_reshaped[:,49,:]

print(input_data.shape)

print(output_data.shape)

torch.Size([200, 49, 3])
torch.Size([200, 3])


In [3]:
import torch.nn as nn
class custom_RNN(nn.Module):
  def __init__(self, input_size, hidden_size, output_size):
    super().__init__()
    self.hidden_size=hidden_size
    #self.hidden_state_old=nn.Parameter(torch.randn(hidden_size))
    self.hidden_to_hidden=nn.Parameter(torch.randn(hidden_size, hidden_size))
    self.input_to_hidden=nn.Parameter(torch.randn(hidden_size, input_size))
    self.bias_input_hidden=nn.Parameter(torch.randn(hidden_size))
    self.bias_hidden_hidden=nn.Parameter(torch.randn(hidden_size))

    self.bias_output=nn.Parameter(torch.randn(output_size))
    self.hidden_to_output=nn.Parameter(torch.randn(hidden_size, output_size))
  def forward(self, x):

    hidden_state_initial=torch.zeros(x.shape[0],self.hidden_size)

    for i in range(x.shape[-2]):
      new_hidden_state=torch.tanh( x[:,i,:] @ self.input_to_hidden.T +self.bias_input_hidden+hidden_state_initial @ self.hidden_to_hidden +self.bias_hidden_hidden)
      hidden_state_initial=new_hidden_state
    output_state=hidden_state_initial @ self.hidden_to_output + self.bias_output
    return output_state

In [13]:
model=custom_RNN(input_size=3, hidden_size=32, output_size=3)
loss=nn.MSELoss()
optimizer=torch.optim.Adam(model.parameters(), lr=0.001)
epochs=10000
for i in range(epochs):

  optimizer.zero_grad()
  output_prediction=model(input_data)
  compute_loss=loss(output_prediction, output_data)
#optimizer.zero_grad()
  compute_loss.backward()
  optimizer.step()
  if i % 100==0:
    print(compute_loss.item())

282.1914367675781
222.10414123535156
179.84771728515625
147.630615234375
121.63566589355469
100.2967300415039
82.9362564086914
68.85294342041016
57.35206985473633
48.45464324951172
41.64460754394531
36.39578628540039
32.26477813720703
29.187204360961914
27.40288734436035
25.385778427124023
23.444841384887695
22.25459098815918
21.386606216430664
20.70927619934082
20.156476974487305
19.68596839904785
19.25824737548828
18.595767974853516
18.212356567382812
17.856178283691406
17.521268844604492
17.204368591308594
16.898107528686523
16.599361419677734
16.30415916442871
16.017019271850586
15.736279487609863
15.460213661193848
15.184370040893555
14.91080093383789
14.651147842407227
14.39721393585205
14.011930465698242
13.768149375915527
13.523038864135742
13.676384925842285
13.562841415405273
13.204330444335938
12.866341590881348
12.498952865600586
12.186692237854004
11.9044828414917
11.625553131103516
11.350706100463867
11.080572128295898
10.815217971801758
10.541234970092773
10.287867546081